# 02 — Interaction Edgelist (Small-Scale Testing)

Loads GloBI plant-pollinator interactions and filters to the 50 target plant species.
Used for initial go/no-go testing only.

**Cleaning steps:**
1. Filter to the 50 target plant species and CONUS bounding box
2. Keep only flower-visitation interaction types
3. Remove 15 self-paired artifacts (same species appearing as both plant and pollinator) → 39 edges
4. Remove 18 plant species mislabeled as pollinators in the pollinator column, and 1 biologically implausible pair (coho salmon) → **18 edges**

**Output:** `plant_pollinator_edges.csv` (18 clean interaction edges)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
BASE        = Path("/scratch/ariana.l")
GLOBIPATH   = BASE / "CfE2026CVforEcology" / "rawpollinatordata" / "interactions.csv.gz"
OUT_DIR     = BASE / "CfE2026CVforEcology" / "rawpollinatordata"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ───────────────────────────────────────────────────────────────
# CONUS bounding box
LAT_MIN, LAT_MAX = 24.0, 49.5
LON_MIN, LON_MAX = -125.0, -66.0

# 7 broad flower-visitation interaction types
INTERACTION_TYPES = [
    "visits flowers of",
    "visited by",
    "pollinates",
    "pollinated by",
    "has flower-visiting",
    "flower-visiting",
    "interacts with",
]

# Top 50 target plant species
TARGET_PLANTS = [
    "Asimina triloba", "Sanguinaria canadensis", "Mitchella repens",
    "Cypripedium acaule", "Erodium cicutarium", "Malosma laurina",
    "Diospyros virginiana", "Dipterostemon capitatus", "Phytolacca americana",
    "Trillium grandiflorum", "Claytonia virginica", "Microstegium vimineum",
    "Trillium ovatum", "Lonicera maackii", "Asclepias syriaca",
    "Rhus glabra", "Ligustrum sinense", "Aquilegia canadensis",
    "Passiflora incarnata", "Chimaphila maculata", "Trillium erectum",
    "Alliaria petiolata", "Celastrus orbiculatus", "Amphicarpaea bracteata",
    "Arisaema triphyllum", "Bignonia capreolata", "Houstonia caerulea",
    "Dicentra cucullaria", "Impatiens capensis", "Conium maculatum",
    "Mertensia virginica", "Convolvulus arvensis", "Glechoma hederacea",
    "Erythronium americanum", "Galium aparine", "Achillea millefolium",
    "Lysimachia borealis", "Caltha palustris", "Sambucus canadensis",
    "Staphylea trifolia", "Kalmia latifolia", "Maianthemum racemosum",
    "Lamium purpureum", "Triteleia laxa", "Erigeron philadelphicus",
    "Chamaenerion angustifolium", "Bellis perennis", "Eschscholzia californica",
    "Sambucus cerulea", "Larrea tridentata"
]

print("Paths OK")

In [ ]:
# Load GloBI interactions
print("Loading GloBI interactions...")
df = pd.read_csv(
    GLOBIPATH,
    usecols=["sourceTaxonName", "targetTaxonName", "interactionTypeName",
             "decimalLatitude", "decimalLongitude"],
    low_memory=False
)
print(f"  Total records: {len(df):,}")
df.head()

In [ ]:
# Filter to target plants, CONUS, and flower-visitation interaction types
df = df.rename(columns={
    "sourceTaxonName": "plant_species",
    "targetTaxonName": "pollinator_species",
    "interactionTypeName": "interaction_type",
    "decimalLatitude": "lat",
    "decimalLongitude": "lon"
})

df = df[
    (df["plant_species"].isin(TARGET_PLANTS)) &
    (df["interaction_type"].isin(INTERACTION_TYPES)) &
    (df["lat"] >= LAT_MIN) & (df["lat"] <= LAT_MAX) &
    (df["lon"] >= LON_MIN) & (df["lon"] <= LON_MAX)
].dropna(subset=["plant_species", "pollinator_species"])

print(f"  After filtering: {len(df):,} records")

In [ ]:
# Deduplicate to unique (plant, pollinator) pairs
edges = df[["plant_species", "pollinator_species"]].drop_duplicates().reset_index(drop=True)
print(f"  Unique edges (raw): {len(edges)}")

# Step 1: Remove self-paired artifacts
edges = edges[edges["plant_species"] != edges["pollinator_species"]]
print(f"  After removing self-pairs: {len(edges)} edges")

In [ ]:
# Step 2: Remove plant species mislabeled as pollinators
# Any species appearing in TARGET_PLANTS should not appear in the pollinator column
edges = edges[~edges["pollinator_species"].isin(TARGET_PLANTS)]
print(f"  After removing mislabeled plant-as-pollinator entries: {len(edges)} edges")

# Step 3: Remove biologically implausible pairs
# Oncorhynchus kisutch (coho salmon) is not a pollinator
edges = edges[edges["pollinator_species"] != "Oncorhynchus kisutch"]
print(f"  After removing implausible pairs: {len(edges)} edges")

edges.head(20)

In [ ]:
# Save
out_path = OUT_DIR / "plant_pollinator_edges.csv"
edges.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
print(f"Final edge count: {len(edges)}")